# Lab 25: LLM Tool Calling and API Integration

**Duration:** 3 hours 30 minutes. **Level:** intermediate. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
A language model only writes text. It cannot read a sensor, switch a light or ask a web service. **Tool calling** gives it hands: you describe your Python functions to the model, and when a question needs one, the model does not answer but **asks your program to run the function** with the right arguments. Your program runs it, gives the result back, and the model writes the final answer. In this lab you build the whole loop around a small simulated smart building (sensors, lights, fans, a door) and a simulated weather service, check every request before it runs, and measure how often the 2B model chooses the right tool.

**By the end of this lab you can:**
1. Explain tool calling: the model **requests** a function, your code **runs** it, and the result goes back to the model.
2. Describe a Python function to the model as a tool (name, description and arguments in JSON).
3. Build the tool loop: ask, run the requested tools, return the results, get the final answer.
4. Validate the arguments of a request, and return errors to the model so it can correct itself.
5. Call a local HTTP API from a tool, handle its failures, and measure how often the model picks the right tool.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **Safety.** The building in this lab is simulated, so nothing real is switched. In real projects (Lab 28 and later) a wrong tool call can open a door or run a motor. That is why the program, and not the model, checks every request. You build that check in Part D.

## 2. Background

**What the model can and cannot do.** A language model reads text and writes text. It does not run code. When you give it a list of **tools**, it can answer with a special message that says: *please call `get_temperature` with `room = kitchen`*. That message is only a **request**. Your program decides whether to run it.

| Step | Who does it | What happens |
|---|---|---|
| 1 | Your program | Sends the question and the list of tools to the model |
| 2 | The model | Answers with a tool request (name and arguments), or with a normal answer |
| 3 | Your program | Checks the request and **runs** the function |
| 4 | Your program | Sends the result back as a message with the role `tool` |
| 5 | The model | Writes the final answer from the result |

Steps 2 to 4 can repeat: the model may ask for several tools in one answer (**parallel calls**), or ask for another tool after it saw the first result.

**Describing a tool.** A tool description is a JSON object with the **name**, a short **description** (the model reads this to decide when to use the tool), and the **parameters**: a name, a type and (optionally) a description for each argument. A good description is like a good prompt (Lab 20): short, exact, with an example.

**Tool calling is not an API call.** These two ideas are often mixed:

| | Tool calling | API integration |
|---|---|---|
| What it is | The model asks your program to run a function | Your program talks to another service over the network |
| Who talks to whom | Model and your program | Your program and a web service |
| In this lab | Part B to D | Part E: a tool that calls a weather service with HTTP |

A tool can run a Python function, or it can call an API. The model does not know which.

**Why check the request?** The model can invent an argument (a room that does not exist), forget one, or ask for a tool you did not give it. Your program must check the **name**, the **arguments** and their **types**, and return a clear error text. The model reads the error and usually corrects itself.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(25)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (20 min)

### Step 1: The Language Model Server
Tool calling needs the language model server of Lab 20 (started with `--jinja`, as in that lab). The cell checks it. If it says `False`, ask the student who started the server in Lab 20 to run `labview.llm_start()`.

In [ ]:
import inspect
import json
import time

import labview
import sensor_tools as building

print("Language model server ready:", labview.llm_ready())
print("Model:", labview.llm_current())

### Step 2: The Simulated Building
`sensor_tools.py` (in the folder of the course, next to `labview.py`) is a small simulated smart building. It has rooms with a temperature and a humidity, lights, fans and doors. The cell shows the code of two functions and reads all the rooms. Everything is in memory: when you run `building.reset()`, the building goes back to the start.

In [ ]:
print(inspect.getsource(building.get_temperature))
print(inspect.getsource(building.set_light))
for room in building.list_rooms():
    print(f"{room:<8} {building.get_temperature(room)} C   {building.get_humidity(room)} %")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which rooms exist, and which room is the warmest:
- What does set_light do to the building (look at its code):
- Why does the first line of the docstring matter for tool calling (read Part B first if you are not sure):

---
# Part B: The Model Asks for a Tool (40 min)

### Step 1: Describe Two Tools by Hand
A tool description is JSON. Read it, then run the cell. The `description` is what the model reads to decide when to use the tool.

In [ ]:
TEMPERATURE_TOOL = {
    "type": "function",
    "function": {
        "name": "get_temperature",
        "description": "Read the temperature of a room in degrees Celsius.",
        "parameters": {
            "type": "object",
            "properties": {"room": {"type": "string", "description": "the room name, for example kitchen or lab"}},
            "required": ["room"],
        },
    },
}

LIGHT_TOOL = {
    "type": "function",
    "function": {
        "name": "set_light",
        "description": "Switch the light of a room on (true) or off (false).",
        "parameters": {
            "type": "object",
            "properties": {"room": {"type": "string"}, "on": {"type": "boolean"}},
            "required": ["room", "on"],
        },
    },
}

TWO_TOOLS = [TEMPERATURE_TOOL, LIGHT_TOOL]
print(json.dumps(LIGHT_TOOL, indent=2))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which parts of the description does the model read to choose a tool:
- What does 'required' mean:
- What type does the argument on have, and what would the model write for it:

### Step 2: The First Request
`labview.llm_message` sends the conversation and the list of tools, and returns the **whole answer** (not streamed). Look at the answer: it has no text. It has a `tool_calls` list.

In [ ]:
SYSTEM = "You help in a small smart building. Use the tools to read sensors and to switch things. Answer in one or two short sentences."

answer = labview.llm_message([{"role": "system", "content": SYSTEM},
                              {"role": "user", "content": "What is the temperature in the kitchen?"}], tools=TWO_TOOLS)
print("finish reason:", answer["finish"], "| time:", round(answer["total_s"], 1), "s")
print(json.dumps(answer["message"].get("tool_calls"), indent=2))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The finish reason and what it tells you:
- The name of the tool and the arguments the model chose:
- Was the answer text empty? Why:

### Step 3: Which Tool for Which Question?
The cell sends six questions with the same two tools and prints what the model asked for. Some questions need no tool at all.

In [ ]:
def requested(question, tools):
    """The first answer of the model: a list of (tool name, arguments), or the text if it asked for no tool."""
    reply = labview.llm_message([{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}], tools=tools)["message"]
    calls = reply.get("tool_calls") or []
    if calls:
        return [(c["function"]["name"], json.loads(c["function"]["arguments"])) for c in calls]
    return (reply.get("content") or "").strip()


QUESTIONS = ["What is the temperature in the kitchen?", "Turn off the light in the lab.", "Switch on the kitchen light and tell me how warm it is there.",
             "Who wrote Hamlet?", "Is it cold in the lab?", "How humid is the garage?"]
for question in QUESTIONS:
    print(f"{question}\n   -> {requested(question, TWO_TOOLS)}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which questions got a tool request, and which got a normal answer:
- What did the model do with the question about humidity? There is no humidity tool in this list:
- Which question asked for two tools in one answer:

### Step 4: The Model Did Not Run Anything
A tool request is only text. The cell asks the model to switch on the kitchen light, and then looks at the building. Is the light on?

In [ ]:
building.reset()
print("Light in the kitchen before:", building.STATE["kitchen"]["light"])
print(requested("Please switch on the kitchen light.", TWO_TOOLS))
print("Light in the kitchen after the request:", building.STATE["kitchen"]["light"])

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Was the light switched on by the request:
- Who has to run the function:
- Why is this good for safety:

---
# Part C: The Tool Loop (45 min)

### Step 1: Describe All Tools Automatically
Writing JSON by hand is slow and easy to get wrong. `building.schema(function)` makes the description from the Python function: its name, the first line of its docstring and its typed arguments. The cell describes all six functions of the building.

In [ ]:
TOOLS = building.all_schemas()
for tool in TOOLS:
    f = tool["function"]
    print(f"{f['name']:<16} {list(f['parameters']['properties'])}   {f['description']}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many tools there are:
- Which function has no argument:
- What happens to the description if you remove the docstring of a function (try it in a copy):

### Step 2: Run a Tool Request
`run_tool` is the part that **your program** owns: it finds the function by name and calls it with the arguments. The result is turned into text with `json.dumps`. Read the cell, then run it.

In [ ]:
def run_tool(name, arguments):
    """Run ONE tool request from the model. Returns the result as text."""
    function = building.FUNCTIONS[name]
    return json.dumps(function(**arguments))


print(run_tool("get_temperature", {"room": "kitchen"}))
print(run_tool("list_rooms", {}))

### Step 3: The Loop
`ask` is the loop of the table in the Background. It asks the model, runs every requested tool, adds the results to the conversation as `tool` messages, and asks again, until the model answers with text. `max_rounds` stops it if the model never stops asking.

In [ ]:
def ask(question, tools=None, runner=None, max_rounds=5, show=True):
    """Ask a question. Returns (final answer, the whole conversation)."""
    tools = TOOLS if tools is None else tools
    runner = runner or run_tool
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for round_number in range(1, max_rounds + 1):
        message = labview.llm_message(messages, tools=tools)["message"]
        messages.append(message)
        calls = message.get("tool_calls") or []
        if not calls:
            return (message.get("content") or "").strip(), messages
        for call in calls:
            name = call["function"]["name"]
            arguments = json.loads(call["function"]["arguments"])
            result = runner(name, arguments)
            if show:
                print(f"  round {round_number}: {name}({arguments}) -> {result}")
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
    return "(stopped: the model asked for tools too many times)", messages


building.reset()
answer, conversation = ask("What is the temperature in the kitchen?")
print("ANSWER:", answer)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The tool the model asked for and the answer it wrote:
- How many times was the model called (rounds + 1):
- What would the answer be without the tool:

### Step 4: More Questions
Several tools in one answer, a tool that changes the building, and a question that needs no tool. Watch the rounds.

In [ ]:
building.reset()
for question in ["Switch on the kitchen light and tell me how warm it is there.", "What are the temperatures in the lab and in the garage?", "Who wrote Hamlet?"]:
    print("Q:", question)
    answer, _ = ask(question)
    print("A:", answer, "\n")
print("Kitchen light is now:", building.STATE["kitchen"]["light"])

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which tools were run for 'What are the temperatures in the lab and in the garage' and in how many rounds:
- What happened with the Hamlet question:
- Is the kitchen light on now? Who switched it on:

---
# Part D: Check the Request, Return the Error (35 min)

### Step 1: Break It on Purpose
The model can ask for a room that does not exist. The cell sends fake requests (as if the model had made them) to `run_tool`. Look at what happens.

In [ ]:
bad_requests = [("get_temperature", {"room": "moon base"}), ("get_temperature", {}), ("get_temperature", {"room": 5}),
                ("open_the_pod_bay_doors", {}), ("set_light", {"room": "lab", "on": "maybe"})]
for name, arguments in bad_requests:
    try:
        print(f"{name}({arguments}) -> {run_tool(name, arguments)}")
    except Exception as error:
        print(f"{name}({arguments}) -> CRASH: {type(error).__name__}: {error}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which fake requests crashed the program and which gave a wrong result:
- What would a crash do to the conversation of a real assistant:

### Step 2: A Safe Runner
`validate` checks the **name**, the **names of the arguments** and the **types** before anything runs. `run_tool_safe` returns a clear error text instead of crashing. The model sees the text as the result of the tool, and can fix its mistake.

In [ ]:
def validate(name, arguments):
    """Check a tool request. Returns a list of problems (empty = fine)."""
    if name not in building.FUNCTIONS:
        return [f"there is no tool '{name}'. The tools are: {', '.join(building.FUNCTIONS)}"]
    parameters = inspect.signature(building.FUNCTIONS[name]).parameters
    problems = [f"missing argument '{p}'" for p in parameters if p not in arguments]
    problems += [f"unknown argument '{a}'" for a in arguments if a not in parameters]
    for p, value in arguments.items():
        if p in parameters:
            expected = parameters[p].annotation
            if not isinstance(value, expected) or (expected is not bool and isinstance(value, bool)):
                problems.append(f"argument '{p}' must be {expected.__name__}, not {type(value).__name__}")
    return problems


def run_tool_safe(name, arguments):
    problems = validate(name, arguments)
    if problems:
        return "ERROR: " + "; ".join(problems)
    try:
        return json.dumps(building.FUNCTIONS[name](**arguments))
    except Exception as error:                      # for example an unknown room
        return f"ERROR: {error}"


for name, arguments in bad_requests:
    print(f"{name}({arguments}) -> {run_tool_safe(name, arguments)}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which problems does validate find, and which does the try-except find:
- Why does the error text name the rooms or the tools that exist:

### Step 3: Let the Model Correct Itself
The cell asks for a room that does not exist. With `run_tool_safe`, the model gets the error text and can answer sensibly.

In [ ]:
building.reset()
answer, conversation = ask("What is the temperature in the moon base?", runner=run_tool_safe)
print("ANSWER:", answer)
answer, conversation = ask("Switch on the lamp in the lab and the garage.", runner=run_tool_safe)
print("ANSWER:", answer)
answer, conversation = ask("Which room is the warmest?", runner=run_tool_safe)
print("ANSWER:", answer)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What did the model do after the error text:
- Did it invent a temperature for the moon base? What does that tell you about checks:
- For the question about the warmest room: which tools did the model use, in how many rounds, and was the answer right:

---
# Part E: A Tool That Calls an API (30 min)

### Step 1: A Simulated Weather Service
A real weather service is a web server that answers HTTP requests with JSON. The cell starts a small one **on this Jetson** (it uses a free port chosen by the system, so your team mates do not clash with you). It knows three cities.

In [ ]:
import threading
import urllib.error
import urllib.parse
import urllib.request
from http.server import BaseHTTPRequestHandler, HTTPServer

WEATHER = {"kuala lumpur": {"temp_c": 31, "rain": True}, "penang": {"temp_c": 30, "rain": False}, "johor bahru": {"temp_c": 29, "rain": True}}


class WeatherAPI(BaseHTTPRequestHandler):
    def do_GET(self):
        url = urllib.parse.urlparse(self.path)
        city = urllib.parse.parse_qs(url.query).get("city", [""])[0].lower()
        if url.path != "/api/weather":
            return self.reply(404, {"error": "no such address"})
        if city not in WEATHER:
            return self.reply(404, {"error": f"no data for '{city}'"})
        self.reply(200, {"city": city, **WEATHER[city]})

    def reply(self, code, data):
        body = json.dumps(data).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, *args):
        pass


api_server = HTTPServer(("127.0.0.1", 0), WeatherAPI)
API = f"http://127.0.0.1:{api_server.server_port}"
threading.Thread(target=api_server.serve_forever, daemon=True).start()
print("Weather service at", API)
print(urllib.request.urlopen(API + "/api/weather?city=Penang", timeout=3).read().decode())

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What does the service answer for Penang:
- What would you see for a city that is not in the list (try it in a new cell):

### Step 2: The Tool
`get_weather` is an ordinary tool function. Inside, it sends an **HTTP request** with a **time-out** and handles the failures: a city with no data, and a service that does not answer. Both come back as an `error` the model can read. Then the function is added to the tools.

In [ ]:
def get_weather(city: str) -> dict:
    """Get the weather outside in a city from the weather service: temperature in Celsius and whether it rains."""
    url = f"{API}/api/weather?city=" + urllib.parse.quote(city)
    try:
        with urllib.request.urlopen(url, timeout=3) as reply:
            return json.loads(reply.read())
    except urllib.error.HTTPError as error:
        return {"error": json.loads(error.read())["error"]}
    except Exception as error:
        return {"error": f"the weather service did not answer ({error})"}


building.FUNCTIONS["get_weather"] = get_weather
TOOLS = building.all_schemas()
print([t["function"]["name"] for t in TOOLS])

building.reset()
answer, _ = ask("Is it raining in Penang, and is it warmer outside than in the kitchen?", runner=run_tool_safe)
print("ANSWER:", answer)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which two tools did the model use:
- Is the weather answer correct? Compare it with the WEATHER dictionary:
- Where is the API call: in the model or in your function:

### Step 3: When the Service Fails
The cell stops the weather service and asks again. The tool has to handle the failure, and the model has to tell you honestly.

In [ ]:
api_server.shutdown()
api_server.server_close()
answer, _ = ask("What is the weather in Kuala Lumpur?", runner=run_tool_safe)
print("ANSWER:", answer)
answer, _ = ask("What is the weather in Atlantis?", runner=run_tool_safe)
print("ANSWER (the city is not known; the service is also stopped):", answer)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What did the tool return when the service was stopped:
- Did the model make up a weather report? How can you stop that (hint: Lab 20, system prompt):

---
# Part F: Measure, Improve and Present (35 min)

### Step 1: How Often Is the Right Tool Chosen?
The cell sends eight questions and compares the tools the model **asks for** (first round only, nothing is run) with the tools it **should** ask for. Start the weather service again first.

In [ ]:
api_server = HTTPServer(("127.0.0.1", 0), WeatherAPI)
API = f"http://127.0.0.1:{api_server.server_port}"
threading.Thread(target=api_server.serve_forever, daemon=True).start()

TESTS = [("What is the temperature in the lab?", {"get_temperature"}),
         ("Turn off the lab light.", {"set_light"}),
         ("Which rooms do you know?", {"list_rooms"}),
         ("Who wrote Hamlet?", set()),
         ("How humid is the garage?", {"get_humidity"}),
         ("Is it raining in Penang?", {"get_weather"}),
         ("Switch on the kitchen light and tell me its temperature.", {"set_light", "get_temperature"}),
         ("What is 17 times 23?", set())]

right = 0
for question, expected in TESTS:
    got = requested(question, TOOLS)
    names = {name for name, _ in got} if isinstance(got, list) else set()
    ok = names == expected
    right += ok
    print(f"{'ok   ' if ok else 'WRONG'} {question:<60} expected {sorted(expected)}  got {sorted(names)}")
print(f"\n{right} of {len(TESTS)} right")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Score:
- Which question was wrong, and what did the model ask for:
- What would you change in the description of a tool to fix it:

### Step 2: Make It Your Own
Add **at least one improvement** from the list, or one of your own. Change the code in the cells above, run them again, then run the test of Step 1 again.

- Add a new tool: `get_pressure(room)` or `list_open_doors()`. Write the function with a docstring, add it to `building.FUNCTIONS`, run `building.all_schemas()` again and ask a question that needs it.
- Make `validate` check that `room` is one of the rooms and reply with the list of rooms.
- Keep a log: write every tool request and its result in a CSV file with the time.
- Add a rule: a tool request for `open_door` is **refused** unless the user says "please" (a first idea of an approval gate; Lab 27 does it properly).
- Write a better description for a tool the model used wrongly and test again.

> **Tip.** Change one thing at a time and run it after each change.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- Which lines did you change:
- Did the score of Step 1 change?:

### Step 3: Report and Present
Fill in the report. Then give a **3-minute demonstration**: show a tool request (the raw JSON), the loop answering a question with two tools, a request that is refused by `validate`, the API tool, and the score of the test.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Tools you have and which one calls an API:
- The score of the test (right of 8):
- One request that `validate` stopped, and the error text:
- Your improvement:
- One problem you found and how you fixed it:
- One idea to make it better in future (hint: Lab 26 and Lab 27):

> **Where next.** You wrote the glue between the model and the functions yourself. In Lab 26 you meet the **Model Context Protocol (MCP)**, a standard way to offer tools to any model, so you do not write this glue again. In Lab 27 the loop becomes an **agent** that works through several steps toward a goal.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The raw tool request (JSON) of the model is shown and explained | |
| 2 | A tool description is written by hand and made by `building.schema` | |
| 3 | The tool loop answers a question with one tool and a question with two tools | |
| 4 | A question that needs no tool is answered without a tool request | |
| 5 | The fake bad requests are stopped by `validate` and the model corrects itself | |
| 6 | The weather tool works, and the failure of the service is handled | |
| 7 | The score of the test of eight questions is shown | |
| 8 | At least one improvement works and you explain the code you changed | |
| 9 | The report is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. Who runs the function in tool calling: the model or your program? Why does this matter for safety?:
- 2. What is in a tool description, and which part decides whether the model picks the tool?:
- 3. What is a parallel tool call? Give an example from this lab.:
- 4. What does the message with the role `tool` contain, and why is it sent back to the model?:
- 5. Name three things `validate` checks before a request runs.:
- 6. Why does the error text name the rooms that exist?:
- 7. What is the difference between tool calling and API integration?:
- 8. The weather service does not answer. What should the tool return, and what should the model say?:
- 9. The model asks for the same tool again and again. Which setting of `ask` protects you?:
- 10. Name one tool you should never give to a model without an approval step. Why?:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `Language model server ready: False` | The server of Lab 20 is not started | Ask the student who started it in Lab 20 to run `labview.llm_start()` |
| `Cannot reach the language model` | The server is not running | The same as above |
| The model answers with text and never asks for a tool | The question is not about a tool, or the description is weak | Make the description of the tool clearer; check that `tools=` is passed |
| `KeyError: 'get_humidity'` or a similar name | The model asked for a tool that is not in `building.FUNCTIONS` | Use `run_tool_safe`, which returns an error text instead of crashing |
| `TypeError: ... unexpected keyword argument` | The model used a wrong argument name | Use `run_tool_safe` and check the descriptions of the tools |
| The answer is slow (more than 5 seconds) | Several rounds, or the long conversation | Look at the rounds printed by `ask`; keep tool results short |
| `Address already in use` or the weather service does not start | An old copy of the service still runs | Run the cell again (it picks a free port), or restart the kernel |
| The test score changes every time | The model is not fully steady | The loop uses temperature 0; if your own code changes it, set it back to 0 |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(25)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- llama.cpp, function calling: https://github.com/ggml-org/llama.cpp/blob/master/docs/function-calling.md
- JSON Schema, the language of the parameters: https://json-schema.org/understanding-json-schema/
- Python, `http.server`: https://docs.python.org/3/library/http.server.html